In [1]:
import pandas as pd

KAGGLE_PATH = '/kaggle/input/listings2021-22/listings_CLEANED_FINAL.csv'

try:
    # Φόρτωση του αρχείου
    df_2021= pd.read_csv(KAGGLE_PATH)
    print("Το αρχείο φορτώθηκε επιτυχώς!")
except FileNotFoundError:
    print(f"Σφάλμα: Το αρχείο δεν βρέθηκε. Ελέγξτε τη διαδρομή: {KAGGLE_PATH}")

Το αρχείο φορτώθηκε επιτυχώς!


In [2]:
# 1. Έλεγχος του τύπου δεδομένων (ΠΡΕΠΕΙ να είναι float64)
print(f"Τύπος δεδομένων στήλης 'price': {df_2021['price'].dtype}")

# 2. Έλεγχος των πρώτων τιμών (ΠΡΕΠΕΙ να είναι καθαροί αριθμοί χωρίς $)
print("Πρώτες 5 τιμές της στήλης 'price':")
print(df_2021['price'].head())

Τύπος δεδομένων στήλης 'price': object
Πρώτες 5 τιμές της στήλης 'price':
0    $100.00
1     $59.00
2     $45.00
3     $60.00
4     $69.00
Name: price, dtype: object


In [3]:
# 1. Αφαιρούμε το σύμβολο $ και τυχόν κόμματα χιλιάδων
# Χρησιμοποιούμε τη στήλη 'price' του DataFrame
df_2021['price'] = df_2021['price'].astype(str).str.replace(r'[$,]', '', regex=True)

# 2. Μετατροπή σε αριθμητικό τύπο (float)
# Το errors='coerce' μετατρέπει τυχόν μη-αριθμητικές τιμές (όπως το 'nan' string) σε πραγματικά NaN.
df_2021['price'] = pd.to_numeric(df_2021['price'], errors='coerce')

print("Η στήλη 'price' καθαρίστηκε ξανά.")

Η στήλη 'price' καθαρίστηκε ξανά.


In [4]:
# Έλεγχος του Τύπου δεδομένων
print(f"Νέος τύπος δεδομένων στήλης 'price': {df_2021['price'].dtype}")

# Έλεγχος των τιμών (πρέπει να είναι αριθμοί)
print("Πρώτες 5 τιμές της καθαρισμένης στήλης 'price':")
print(df_2021['price'].head())

Νέος τύπος δεδομένων στήλης 'price': float64
Πρώτες 5 τιμές της καθαρισμένης στήλης 'price':
0    100.0
1     59.0
2     45.0
3     60.0
4     69.0
Name: price, dtype: float64


In [5]:
# 1. Βρίσκουμε τον συνολικό αριθμό NaN ανά στήλη
print("Αριθμός NaN ανά Στήλη (Απολυτός Αριθμός):")
nan_counts = df_2021.isnull().sum()
# Εμφανίζει μόνο τις στήλες με NaN, ταξινομημένες
print(nan_counts[nan_counts > 0].sort_values(ascending=False)) 

print("\n" + "="*50) # Διαχωριστικό

# 2. Βρίσκουμε το ποσοστό NaN ανά στήλη
print("Ποσοστό NaN ανά Στήλη (ως % των 9640 γραμμών):")
nan_percentage = df_2021.isnull().mean() * 100
# Εμφανίζει μόνο τις στήλες με NaN, ταξινομημένες κατά ποσοστό
print(nan_percentage[nan_percentage > 0].sort_values(ascending=False))

Αριθμός NaN ανά Στήλη (Απολυτός Αριθμός):
neighbourhood_group_cleansed    9640
bathrooms                       9640
calendar_updated                9640
host_about                      4198
neighborhood_overview           2920
neighbourhood                   2920
license                         2364
host_neighbourhood              2320
review_scores_value             2219
review_scores_checkin           2218
review_scores_location          2218
review_scores_communication     2217
review_scores_cleanliness       2217
review_scores_accuracy          2217
review_scores_rating            2177
first_review                    2177
last_review                     2177
reviews_per_month               2177
host_acceptance_rate            2066
host_response_rate              1871
host_response_time              1871
bedrooms                         814
description                      108
beds                              66
host_location                     29
host_has_profile_pic             

In [6]:
columns_to_drop = [
    'calendar_updated', 
    'neighbourhood_group_cleansed', 
    'host_neighbourhood',
    'host_about',
    'neighborhood_overview',
    'license',
    'bathrooms' # Θα χρησιμοποιήσουμε το 'bathrooms_text'
]

df_2021 = df_2021.drop(columns=columns_to_drop, axis=1)
print(f"Διαγράφηκαν {len(columns_to_drop)} στήλες.")

Διαγράφηκαν 7 στήλες.


In [7]:
# 1. Διαγραφή γραμμών όπου η ΤΙΜΗ είναι NaN
df_2021= df_2021.dropna(subset=['price'])
print(f"Διαγράφηκαν {1153} γραμμές με NaN στην τιμή. Νέος αριθμός γραμμών: {len(df_2021)}")

# 2. Συμπλήρωση αριθμητικών στηλών με τη Διάμεσο (Median)
numerical_cols_to_fill = ['beds', 'bedrooms']
for col in numerical_cols_to_fill:
    df_2021[col] = df_2021[col].fillna(df_2021[col].median())

# 3. Συμπλήρωση Κριτικών & Ποσοστών με 0
review_cols = [col for col in df_2021.columns if 'review_scores_' in col or 'reviews_per_month' in col or 'rate' in col]
df_2021[review_cols] = df_2021[review_cols].fillna(0)

print("Ολοκληρώθηκε ο αριθμητικός καθαρισμός και η διαγραφή NaN τιμής.")

Διαγράφηκαν 1153 γραμμές με NaN στην τιμή. Νέος αριθμός γραμμών: 9640
Ολοκληρώθηκε ο αριθμητικός καθαρισμός και η διαγραφή NaN τιμής.


In [8]:
df_2021['neighbourhood'] = df_2021['neighbourhood'].fillna('Unknown')
df_2021['host_response_time'] = df_2021['host_response_time'].fillna('Unknown')
df_2021['host_is_superhost'] = df_2021['host_is_superhost'].fillna('f') # 'f' για False

print("Συμπληρώθηκαν οι κατηγορικές στήλες.")

Συμπληρώθηκαν οι κατηγορικές στήλες.


In [9]:
print(df_2021.columns.tolist())

['id', 'listing_url', 'scrape_id', 'last_scraped', 'name', 'description', 'picture_url', 'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'has_availability', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'calendar_last_scraped', 'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d', 'first_review', 'last_review'

In [10]:
OUTPUT_FILE_PATH = '/kaggle/working/listings_cleaned_final.csv'
df_2021.to_csv(OUTPUT_FILE_PATH, index=False)

print(f"Το καθαρισμένο αρχείο αποθηκεύτηκε: {OUTPUT_FILE_PATH}")

Το καθαρισμένο αρχείο αποθηκεύτηκε: /kaggle/working/listings_cleaned_final.csv


In [11]:
df_2021['last_scraped'] = pd.to_datetime(df_2021['last_scraped'], errors='coerce') 
df_2021['ETOS_LISTINGS'] = df_2021['last_scraped'].dt.year 
print("Το df_2021 είναι έτοιμο.")

Το df_2021 είναι έτοιμο.


In [12]:
df_2021['ETOS_LISTINGS'] = pd.to_numeric(df_2021['ETOS_LISTINGS'], errors='coerce').astype('Int64')
df_2021['ETOS_LISTINGS'] = pd.to_numeric(df_2021['ETOS_LISTINGS'], errors='coerce').astype('Int64')

print(f"Τύπος δεδομένων ETOS_LISTINGS (Combined): {df_2021['ETOS_LISTINGS'].dtype}")
print(f"Τύπος δεδομένων ETOS_LISTINGS (Macro): {df_2021['ETOS_LISTINGS'].dtype}")

Τύπος δεδομένων ETOS_LISTINGS (Combined): Int64
Τύπος δεδομένων ETOS_LISTINGS (Macro): Int64
